# 03. Siniestralidad y conglomerados

In [1]:
# ------------------------------------------
# 1. Importación de librerías
# ------------------------------------------

from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

# ------------------------------------------
# 2. Configuración de rutas
# ------------------------------------------

# Localizar la carpeta principal del proyecto
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "funciones_reservas.py").exists()
)

# Permitir importar nuestras funciones
sys.path.insert(0, str(ROOT))

from funciones_reservas import leer_snapshot

# Definir las carpetas de trabajo
PROCESSED = ROOT / "data" / "processed"

# ------------------------------------------
# 3. Cargar las bases ajustadas
# ------------------------------------------

# Recuperar las bases previamente ajustadas a diciembre de 2025
emision_ajustada = leer_snapshot(
    PROCESSED / "emision_ajustada_2025.parquet"
)

siniestros_ajustados = leer_snapshot(
    PROCESSED / "siniestros_ajustados_2025.parquet"
)

# Mostrar las dimensiones de las bases
print(f"Emisión: {emision_ajustada.shape}")
print(f"Siniestros: {siniestros_ajustados.shape}")


Emisión: (21353, 11)
Siniestros: (2763, 14)


# 3.1. Preparación y cálculo de factores de siniestralidad

## 3.1.1. Cálculo de factores de siniestralidad por año de origen

Qué tan grandes son los siniestros en relación con la prima emitida por año

In [2]:
# ------------------------------------------
# 3.1.1 Siniestralidad total por año 
# ------------------------------------------

# Columnas de desarrollo ajustadas por INPC
columnas = [f"{i}_monto pagado_ajustada" for i in range(4)]

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen
siniestros_anual = (
    siniestros_ajustados
    .groupby("año")["siniestro_total"]
    .sum()
).round(2)

# Sumar todas las primas emitidas de cada año
primas_anual = (
    emision_ajustada
    .groupby("año")["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor de siniestralidad del mercado
factor_anual = siniestros_anual / primas_anual.replace(0, 1)

factor_anual.reset_index().rename(columns={0: "factor_siniestralidad_anual"}).set_index("año")

,factor_siniestralidad_anual
año,
2015,1.728247
2016,0.553433
2017,0.336023
2018,0.376379
2019,0.382350
2020,0.638191
2021,0.519178
2022,0.672990
2023,0.442951


In [3]:
# ------------------------------------------
# Métricas de siniestralidad por año
# ------------------------------------------
    
print("Métricas de siniestralidad anual")
factor_anual.describe()


Métricas de siniestralidad anual


count    11.000000
mean      0.611773
std       0.395429
min       0.336023
25%       0.379365
50%       0.519178
75%       0.655591
max       1.728247
dtype: float64

In [4]:
# ------------------------------------------------------
# Evolución del factor de siniestralidad anual por año
# ------------------------------------------------------

fig = px.line(
    factor_anual, 
    x=factor_anual.index,
    y=factor_anual.values,
    markers=True,
    title="Evolución del factor de siniestralidad anual",
    labels={"x": "Año", "y": "Factor de siniestralidad"},
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text = "Indicator"
)

fig.show()

El factor de siniestralidad del mercado presenta fluctuaciones importantes entre 2015 y 2025, destacando 2015 como el año con mayor siniestralidad relativa (1.72), mientras que en los años posteriores los factores se mantienen por debajo de 1, este comportamiento sugiere que la siniestralidad no ha seguido una tendencia constante y que podrían existir diferencias importantes entre los segmentos del mercado, por lo que resulta conveniente analizar sus factores individuales para identificar cuáles presentan mayor siniestralidad y variabilidad.

## 3.1.2. Calculo de factores de siniestralidad por tipo de embarcación y uso (perfil de riesgo) en cada año de origen

Qué tan grandes son lo siniestros de cada perfil de riesgo en razón con la prima emitida

In [5]:
# ------------------------------------------
# 3.1.2 Siniestralidad total por año 
# ------------------------------------------

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen, tipo de embarcación y uso
siniestros_segmento = (
    siniestros_ajustados
    .groupby(["año", "tipo_embarcacion", "uso"])["siniestro_total"]
    .sum()
    .round(2)
)

# Sumar todas las primas emitidas de cada año, tipo de embarcación y uso
primas_anual_tipo = (
    emision_ajustada
    .groupby(["año", "tipo_embarcacion", "uso"])["prima_emitida_ajustada"]
    .sum()
).round(2)

# Calcular el factor de siniestralidad del mercado
factor_anual_tipo = siniestros_segmento / primas_anual_tipo.replace(0, 1)

print("Factor de siniestralidad anual por tipo de embarcación y uso")

factor_anual_tipo.reset_index().rename(columns={0: "factor_siniestralidad_segmento"}).set_index(["año", "tipo_embarcacion", "uso"])

Factor de siniestralidad anual por tipo de embarcación y uso


factor_siniestralidad_segmento
año  tipo_embarcacion uso                                               
2015 abastecedor      abastecedor                                    NaN
                      comercial                                 0.262124
                      otro uso                                  9.549007
                      pesca                                     0.166331
                      privado y/o placer                             NaN
...                                                                  ...
2025 velero           pesca                                          NaN
                      privado y/o placer                        0.912612
                      turistico                                 0.045509
     yate             privado y/o placer                        1.669954
                      turistico                                 0.516606

[885 rows x 1 columns]

In [6]:
# ------------------------------------------
# 3. Identificación de segmentos
# ------------------------------------------

# Convertir las primas agrupadas en un DataFrame
emision_segmentos = primas_anual_tipo.reset_index()

# Identificar combinaciones únicas de tipo y uso
segmentos = emision_segmentos[
    ["tipo_embarcacion", "uso"]
].drop_duplicates()

print("Número de segmentos:", len(segmentos))

# --------------------------------------------------
# Cobertura de segmentos durante 2015-2022
# --------------------------------------------------

# Seleccionar los años con desarrollo completo
periodo = emision_segmentos[
    emision_segmentos["año"].between(2015, 2022)
]

# Contar años con prima positiva por segmento
cobertura_8 = (
    periodo
    .assign(positiva=periodo["prima_emitida_ajustada"] > 0)
    .groupby(["tipo_embarcacion", "uso"])["positiva"]
    .sum()
)

# Identificar segmentos con prima positiva en los 8 años
segmentos_completos = cobertura_8[cobertura_8 == 8].index

print("Segmentos completos 2015-2022:", len(segmentos_completos))

# --------------------------------------------------
# Participación de los segmentos seleccionados
# --------------------------------------------------

# Identificar los 34 segmentos dentro de 2015-2022
seleccion = periodo.set_index(
    ["tipo_embarcacion", "uso"]
).index.isin(segmentos_completos)

# Calcular las primas emitidas del periodo
prima_mercado = periodo["prima_emitida_ajustada"].sum()
prima_segmentos = periodo.loc[
    seleccion, "prima_emitida_ajustada"
].sum()

# Calcular su participación dentro del mercado
participacion = prima_segmentos / prima_mercado * 100

print(f"Participación de los 34 segmentos: {participacion:.2f}%")

Número de segmentos: 132
Segmentos completos 2015-2022: 34
Participación de los 34 segmentos: 89.76%


Para el análisis de conglomerados se consideraron las combinaciones de tipo de embarcación y uso, identificando inicialmente 132 segmentos. Se seleccionaron 34 con primas emitidas positivas durante todos los años de 2015 a 2022, los cuales representan el 89.76% de las primas emitidas del periodo. Esta selección permite trabajar con trayectorias de siniestralidad temporalmente comparables y con una parte importante del mercado.

In [7]:
# ---------------------------------------------------------------------
# Dataframe del factor de siniestralidad de los segmentos seleccionados
# ---------------------------------------------------------------------

# Convertir los factores en DataFrame
df_factores = factor_anual_tipo.rename("factor_siniestralidad").reset_index()

# Identificar los 34 segmentos seleccionados
filtro = df_factores.set_index(
    ["tipo_embarcacion", "uso"]
).index.isin(segmentos_completos)

# Filtrar los segmentos y años de interés
df_34 = df_factores.loc[
    filtro & df_factores["año"].between(2015, 2022)
].copy()

df_34["factor_siniestralidad"] = (df_34["factor_siniestralidad"].fillna(0))

df_34["segmento"] = df_34["tipo_embarcacion"] + " - " + df_34["uso"]

In [8]:
# ------------------------------------------
# Mover cada año a una columna separada su respectivo factor de siniestralidad
# ------------------------------------------

df_34_pivot = df_34.pivot(index="segmento", columns="año", values="factor_siniestralidad")

df_34_pivot.rename(columns={i: f"fs_{i}" for i in range(2015, 2023)}, inplace=True)

df_34_pivot

año,fs_2015,fs_2016,fs_2017,fs_2018,fs_2019,fs_2020,fs_2021,fs_2022
segmento,,,,,,,,
abastecedor - abastecedor,0.000000,3.128291,0.326619,1.448700,0.027616,0.625268,1.854826,0.335932
abastecedor - comercial,0.262124,0.137835,0.157054,0.213426,0.027484,0.514290,0.396312,0.600858
abastecedor - privado y/o placer,0.000000,0.003028,0.000000,0.373122,0.235437,0.000000,0.308010,6.797694
abastecedor - turistico,0.000000,0.000000,0.000000,0.000000,0.000000,0.021670,0.014040,0.000000
atunero - pesca,0.045858,0.267407,0.080078,0.371242,0.264100,2.607975,4.002534,3.446874
barcaza o chalan - abastecedor,0.575019,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
barcaza o chalan - turistico,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
buque-tanque - comercial,35.767174,1.063666,0.705245,3.470587,6.861467,1.761038,0.163708,0.065368
buque-tanque petrolero y productos derivados - comercial,0.000735,0.122723,0.360971,0.182118,0.655586,0.033952,2.038834,0.038514


Se identificaron perfiles de riesgo que, a pesar de presentar primas emitidas, no registraron siniestros durante determinados años, por lo que se les asignó un factor de siniestralidad igual a cero, ya que no existe un monto de siniestros que representar respecto a sus primas.

# 3.2. Preprocesamiento multivariado

## 3.2.1. Formulación del problema 

El objetivo de este apartado es construir conglomerados respecto a perfiles de riesgo con comportamientos parecidos en el timpo. Las variables con las cuales construiremos estos conglomerados son los factores de riesgo de cada año respectivamente. 

In [9]:
# --------------------------------------------
# Matriz de correlación de siniestralidad
# --------------------------------------------

# Base para correlación
X = df_34_pivot.fillna(0).copy()

# Renombrar columnas para que solo aparezca el año
X.columns = [col.replace("fs_", "") for col in X.columns]

# Calcular matriz de correlación
correlation_matrix = X.corr().round(2)

# Heatmap
heatmap_fig = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    color_continuous_scale="PuOr",   # mejor para correlaciones
    zmin=-1,                           # mínimo teórico de correlación
    zmax=1,                            # máximo teórico de correlación
    aspect="auto",
    title="Matriz de correlación de siniestralidad por segmento",
)

# Ajustes visuales
heatmap_fig.update_layout(
    width=800,
    height=700,
    title_x=0.5,
    xaxis_title="Año",
    yaxis_title="Año",
    margin=dict(l=40, r=40, t=80, b=40),
    coloraxis_colorbar=dict(title="Correlación")
)

heatmap_fig.update_xaxes(tickangle=0)
heatmap_fig.update_yaxes(autorange="reversed")

heatmap_fig.show()

# 3.3 Análisis de Valores extremos

Queremos indagar porque la variable de factor de siniestralidad del año 2020 es de 1 respecto con la de 2022

In [10]:
# --------------------------------------------------------------
# ¿Que segmento afecta más la correlación entre 2020 y 2022?
# --------------------------------------------------------------

# Revisar correlación entre 2020 y 2022
revision = X.copy()
revision.columns = revision.columns.astype(str).str.replace(
    "factor_siniestralidad_", "", regex=False
)

print("\nSegmentos con factores más altos en 2020:")
display(
    revision[["2020", "2022"]]
    .sort_values("2020", ascending=False)
    .head(10)
)


Segmentos con factores más altos en 2020:


,2020,2022
segmento,,
otro tipo de embarcacion - comercial,226.424214,2085.703699
pesca multiple - pesca,2.618898,1.130353
atunero - pesca,2.607975,3.446874
dragas - comercial,1.847401,0.000000
buque-tanque - comercial,1.761038,0.065368
velero - turistico,1.344627,2.638522
embarcacion de investigacion - oficial,1.218850,0.000510
ferry-transbordador - turistico,0.991895,0.000000
yate - privado y/o placer,0.783342,0.218107


Los factores extraordinarios del segmento "otro tipo de embarcación – comercial" es por la presencia de siniestros de gran magnitud frente a primas emitidas relativamente pequeñas, particularmente en 2020 y 2022. Este segmento influye considerablemente en la correlación entre ambos años, por lo que será necesario evaluar su efecto sobre los métodos multivariados mediante un análisis de sensibilidad.

In [11]:
# ---------------------------------
# Correlación Spearman
# ---------------------------------

# Calcular matriz de correlación 
correlation_matrix_spm = X.corr(method="spearman").round(2)

# Heatmap
heatmap_fig_spm = px.imshow(
    correlation_matrix_spm,
    text_auto=".2f",
    color_continuous_scale="PuOr",   # mejor para correlaciones
    zmin=-1,                           # mínimo teórico de correlación
    zmax=1,                            # máximo teórico de correlación
    aspect="auto",
    title="Matriz de correlación de siniestralidad por segmento",
)

# Ajustes visuales
heatmap_fig_spm.update_layout(
    width=800,
    height=700,
    title_x=0.5,
    xaxis_title="Año",
    yaxis_title="Año",
    margin=dict(l=40, r=40, t=80, b=40),
    coloraxis_colorbar=dict(title="Correlación")
)

heatmap_fig_spm.update_xaxes(tickangle=0)
heatmap_fig_spm.update_yaxes(autorange="reversed")

heatmap_fig_spm.show()

# 3.4. Análisis de conglomerados


- 3.4.1. Selección de la distancia
- 3.4.2. Aplicación de métodos de agrupación
- 3.4.3. Definición de los valores de \(k\) a evaluar

# 3.5. Evaluación de los conglomerados
- 3.5.1. Silhouette Score
- 3.5.2. Calinski-Harabasz y WCSS
- 3.5.3. Comparación de métodos y selección del número de conglomerados
- 3.5.4. Asignación y caracterización de los grupos

# 3.6. Validación de robustez y estabilidad
- 3.6.1. Comparación de métodos mediante ARI
- 3.6.2. Sensibilidad a observaciones extremas
- 3.6.3. Estabilidad temporal de los conglomerados

# 3.7. Conclusiones
- 3.7.1. Perfiles de siniestralidad identificados
- 3.7.2. Calidad, estabilidad y limitaciones de los conglomerado